# SINDBAD Parameter Hybrid Inversion Tutorial

This notebook is a notebook-style translation of `run_hybrid_inversion.jl`.

The purpose of this workflow is to train a hybrid model inside the SINDBAD framework, use the trained machine-learning model to predict site-level parameter vectors, and then compare default versus hybrid-informed forward simulations. The scientific question is to address the biogeochemical parameters which are difficult to constrain with traditional process-based modeling approaches, but can be learned from data using machine learning. The hypothesis here is that the site-level parameters are finally determined by static covariates like plant function type, mean annual temperature, and mean annual precipitation, soil types...etc. But how can we learn the mapping from static covariates to site-level parameters? This is where machine learning comes in. By training a machine learning model on a set of sites with known parameters and covariates, we can learn the underlying relationships and then apply the model to predict parameters for new sites based on their covariates.

Conceptually, this sits between pure process-based modeling and fully data-driven emulation. The mechanistic model still defines the carbon and water processes, but machine learning is used to learn how parameter values vary across sites from environmental covariates and observational constraints. This is useful because many terrestrial ecosystem parameters are difficult to generalize across climate and biome gradients when estimated independently at each site.

The tutorial below demonstrates this idea for two configurations:

- **WROASTED_HB**, a more complete coupled carbon-water setup, and
- **LUE**, a simpler and faster light-use-efficiency-based setup.

For each case, the notebook:

- loads the experiment settings,
- prepares forcing and observations,
- trains the hybrid model,
- predicts site-specific parameters,
- runs default and hybrid-informed forward simulations, and
- visualizes the improvement in performance.

For the tool, we would use Sindbad.
---
## SINDBAD
[SINDBAD](http://sindbad-mdi.org/) is a model-data integration framework for terrestrial carbon-water processes [[Koirala et al., in prep.](https://essopenarchive.org/users/551954/articles/1271244)]. Is built in Julia with a view on speed and differenciability for the development of representation of processes and responses of ecosystem functioning to meteorological conditions and changes in climate. Sets on the concept of modularity to formaly test hypothesis on the representation of processes / models ($f(X,\theta)$), for given observational constraints ($Y$) and drivers ($X$) of the carbon and water dynamics in terrestrial ecosystems. Modularity is extended to the initial condition problem ($\text{x}^*_0$), cost functions ($\mathcal{L(\theta)}$) and optimization algorithms ($\mathcal{O}$). SINDBAD integrates machine learning for enhancing the representation of processes in mechanistically-inspired models, hybrid modeling [Reichstein et al., 2019], by learning ML-based parameterizations [e.g. Bao et al., 2024], paving way for process abstraction [Son et al., 2024].

---
## WROASTED: a Simple Coupled Carbon–Water Ecosystem Model
The carbon dynamics,  $\frac{dC}{dt}$, are simulated as the difference between gross assimilation and respiratory fluxes
$$
\frac{dC}{dt} = GPP - R_{ECO}
$$

where ${GPP}$, gross primary productivity, results from photosynthetic activity and $R_{ECO}$, ecosystem respiration, is the sum of autotrophic and heterotrophic respiratory fluxes, namely, $R_{A}$ and $R_{H}$. 

$R_{A}$ integrates both maintenance and growth respiration, $R_{M}$ and $R_{G}$, where  $R_{M}$  can be generically written like:

$$
R_{M} =\sum_{i=1}^{N} \tau_i \cdot C_i \cdot f_T 
$$

$i$ representing the different carbon pools ($C_i$) in vegetation - root/wood/leaf/reserves; $\tau_i$ the turnover rate of pool $i$ , and $f_T$ the temperature dependence of metabolic activity, usually a $Q_{10}$ function; while $R_G=Y_G \cdot GPP$, being $Y_G$ and constant growth efficiency parameter [see Amthor, 2001]. 

$R_H$ results from litter and soil decomposition:
$$
R_{H} =\sum_{i=1}^{N} \tau_i \cdot C_i \cdot f_T \cdot f_W
$$

$i$ representing the different heterotrophic carbon pools ($C_i$) in soils - fast and slow litter and organic carbon pools; $\tau_i$ the turnover rate of pool $i$ , $f_T$ and $f_W$ the temperature and soil moisture sensitivity of decomposition function.

Soil moisture dynamics, $\frac{dW}{dt}$:
$$
\frac{dW}{dt} = P_r - E_i - E_s - Q - D - T_r
$$

Being: $Pr$: precipitation; $E_i$: interception evaporation; $E_s$: soil evaporation; $Q$: surface runoff; $D$: drainage; $T_r$: plant transpiration.

Transpiration is tighly coupled to $GPP$, estimated as: 

$$
GPP = min(GPP_D,GPP_S)
$$

Being demand $GPP$:
$$
GPP_S = \epsilon^* \cdot f\text{APAR} \cdot \text{PAR} \cdot (f_L \cdot f_{CI} \cdot f_T \cdot f_{VPD} \cdot f_W)
$$
The product between: maximum light use efficiency, $\epsilon^*$; the fraction of photosynthetically active radiation, $\text{APAR}$, absorbed by leafs, $f\text{APAR}$; and the instantaneous effect of light intensity $f_L$, cloudiness index $f_CI$, vapor pressure deficit $f_VPD$ and soil moisture $f_W$ [see Bao et al., 2023; 2024].

And  supply $GPP$:

$$
GPP_S = PAW^{k_{Tr}} \cdot WUE
$$

Where where the daily variations in water use efficiency, $WUE$, result from changes in $VPD$ and $\quad [CO_2]_{atm}$. Upon $C$ assimilation by vegetation, and deduced $R_A$ costs, the available carbon is transported to the different vegetation pools depending on environmental conditions, as inspired by the growind season index (GSI) model [see Koirala et al., in print; Jolly et al., 2005]. 

Overall, WROASTED includes >40 parameters controlling the responses of carbon and water dynamics in terrestrial ecosystems constrainable by observations of ecosystem fluxes, eddy covariance, plant phenology from remote sensing EO data, and above ground biomass stocks, where available [see Koirala et al., in print].


---
## The challenge
To calibrate and generalize the model parameterization.

---
## Parameter inversion
The goal is to find $\theta$ such that the model predictions $f(X, \theta)$ best match observed datasets $y$. Here, the terrestrial ecosystem model, WROASTED, represented by $f(X, \theta)$, predicts a set of ecosystem carbon and water state and flux variables, $\hat{y}$, observed at locations: 
- $X$: meteorological drivers (i.e., temperature, radiation, precipitation, $VPD$, etc);
- $\theta$: parameter vector to be estimated;
- $y$: observations (e.g., $GPP$, $T_r$, evapotranspiration, $R_{ECO}$, aboveground biomass AGB, $f\text{APAR}$)

### Optimization problem
Generically can be written:
$$
\theta^*=\arg\min_{\theta \in \Theta} \; \mathcal{L}(\theta)\quad\text{via}\quad\mathcal{O}
$$

Where:
- $\mathcal{L}(\theta)$: is the cost function quantifying the mismatch between model predictions and observations,
- $\Theta$: feasible parameter space (e.g., bounds or priors on $\theta$),
- $\mathcal{O}$: optimization operator/algorithm (e.g., gradient descent, L-BFGS, CMA-ES)

In the exercise here, for fluxes and phenology time series, the loss function $\mathcal{L}(\theta)$ is set to the normalized Nash-Sutcliffe Efficiency (NNSE)
$$
\text{NNSE}(\theta) = 1 - \frac{1}{2-NSE}
$$
$$
\text{NSE}(\theta) = 1 - \frac{\sum_{i=1}^{N} (y_i - f(X_i, \theta))^2}{\sum_{i=1}^{N} (y_i - \bar{y})^2}
$$

While for stocks, AGB, an adjusted normalized mean average error is used
$$
NMAE = \frac{\sum_{i=1}^{N} |y_i - f(X_i, \theta)|}{N \cdot (1+ \bar{y})}
$$
$$
\theta^* = \arg\min_{\theta} \; \mathcal{L}(\theta)
$$

Here, a full list of parameter is shown in the following table:

| Parameter                                     | Meaning                                                                                   | Units     |
| --------------------------------------------- | ----------------------------------------------------------------------------------------- | --------- |
| rootMaximumDepth.constant_frac_max_root_depth | Root depth as a fraction of soil depth.                                     |           |
| rootWaterEfficiency.k_efficiency_cVegRoot     | Rate constant of exponential relationship.             | m2/gC     |
| treeFraction.constant_frac_tree               | Constant fraction of vegetation treated as tree cover.                                    |           |
| fAPAR.k_extinction                            | Canopy light extinction coefficient in Beer–Lambert fAPAR formulation.                    |           |
| snowMelt.melt_T                               | Temperature melt factor.                   | mm/°C     |
| snowMelt.melt_Rn                              | Radiation melt factor (controls melt sensitivity to net radiation).                       | mm/(W/m2) |
| runoffSaturationExcess.β                      | Linear scaling parameter to get the berg parameter from vegFrac.                   |           |
| evaporation.α                                 | α coefficient of Priestley-Taylor formula for soil.                                 |           |
| evaporation.k_evaporation                     | Fraction of soil water that can be used for soil evaporation.             |           |
| drainage.dos_exp                              | Exponent of non-linearity for dos influence on drainage in soil.                      |           |
| groundWRecharge.dos_exp                       | Exponent of non-linearity for dos influence on drainage to groundwater.          |           |
| transpirationSupply.k_transpiration           | Fraction of total maximum available water that can be transpired. |           |
| gppPotential.εmax                             | Maximum light-use efficiency for potential GPP.                                           | gC/MJ     |
| gppDiffRadiation.μ                            | Parameter controlling diffuse radiation effect on light-use efficiency.                   |           |
| gppAirT.opt_airT                              | Optimal air temperature for photosynthesis.                                               | °C        |
| gppAirT.opt_airT_A                            | Increasing slope of sensitivity.                       |           |
| gppAirT.opt_airT_B                            | Decreasing slope of sensitivity.                       |           |
| gppVPD.κ                                      | VPD sensitivity parameter reducing photosynthesis at high VPD.       | 1/hPa     |
| gppVPD.c_κ                                    | Scaling/offset parameter for the VPD stress function.                                 |           |
| gppVPD.sat_ambient_CO2                        | Saturated ambient CO₂ saturation level.                      | ppm       |
| gppSoilW.q                                    | Sensitivity of GPP to soil moisture.                       |           |
| gppSoilW.θstar                                | Optimal soil moisture threshold for GPP limitation.                              | m3/m3     |
| WUE.WUE_one_hpa                               | Baseline water using efficiency at 1 hPa VPD (or reference condition).                      |           |
| WUE.κ                                         | Parameter controlling water using efficiency sensitivity to VPD.                            | 1/hPa     |
| cCycleBase.c_τ_Root                           | Root carbon turnover rate.                                             |           |
| cCycleBase.c_τ_Wood                           | Wood/stem carbon turnover rate.                                        |           |
| cCycleBase.c_τ_Leaf                           | Leaf carbon turnover rate.                                             |           |
| cCycleBase.c_τ_LitFast                        | Fast litter turnover rate.                                             |           |
| cCycleBase.c_τ_LitSlow                        | Slow litter turnover rate.                                             |           |
| cCycleBase.c_τ_SoilSlow                       | Slow soil carbon turnover rate.                                        |           |
| cCycleBase.c_τ_SoilOld                        | Old/passive soil carbon turnover rate.                                 |           |
| cCycleBase.ηH                                 | Scaling factor for heterotrophic pools after spinup.           |           |
| cCycleBase.c_remain                           | Remaining carbon after disturbance.               |           |
| cTauSoilT.Q10                                 | Q10.                             |           |
| cTauSoilW.opt_soilW                           |                                     | m3/m3     |
| autoRespirationAirT.Q10                       | Q10 parameter for maintenance respiration.                                 |           |
| autoRespiration.RMN                           | Nitrogen efficiency rate of maintenance respiration.                                                      |           |
| autoRespiration.YG                            | Growth yield coefficient, or growth efficiency. Loosely: (1-YG)*GPP is growth respiration.                                                  |           |
| cFlow.slope_leaf_root_to_reserve              | Allocation slope from leaf/root pools to reserve carbon.                                  |           |
| cFlow.slope_reserve_to_leaf_root              | Allocation slope from reserve carbon to leaf/root growth.                                 |           |
| cFlow.k_shedding                              | Shedding/litterfall rate coefficient (controls biomass shedding).                         | 1/day     |
| cFlow.f_τ                                     | Contribution factor for current stressors.                                |           |


---
## Setting up SINDBAD
Navigate to the [SINDBAD-Tutorials for AI4PEX repository](GitHubLink) and install. Please follow instructions. For us, [VS Code](https://code.visualstudio.com/) has been a very fluid host for [Julia](https://julialang.org/) developments.

### Get the data for these SINDBAD codes
The data can be found within ```../cube_generation/data```.

## Let's go...
### Get packages and goodies to go.

## 1. Activate the environment and load packages

The notebook uses the `SindbadTutorials` environment together with the machine-learning and plotting tools needed by the hybrid workflow.

In [ ]:
using Revise
using SindbadTutorials
using Sindbad.MachineLearning
using Sindbad.MachineLearning.Random
using SindbadTutorials.Plots
using Flux, PreallocationTools, FiniteDiff, FiniteDifferences, ForwardDiff, Optimisers

project_root = isdir(joinpath(pwd(), "tutorials")) ? pwd() : abspath(joinpath(pwd(), "..", ".."))
hybrid_dir = joinpath(project_root, "tutorials", "hybrid_inversion")
wroasted_setup_dir = joinpath(project_root, "tutorials", "setups", "WROASTED_HB")
lue_setup_dir = joinpath(project_root, "tutorials", "setups", "LUE")

## 2. Select the sites used for training and diagnostics

By default, all hybrid-eligible sites are used. The optional `do_random` flag allows you to test the workflow with a smaller random subset, which can be useful for debugging or for quick notebook runs.

In [ ]:
selected_site_indices = getSiteIndicesForHybrid()
do_random = 0

if do_random > 0
    Random.seed!(1234)
    selected_site_indices = first(shuffle(selected_site_indices), do_random)
end

path_output = ""

## 3. Helper functions

The original script repeats the same pattern for the WROASTED and LUE configurations. Here that logic is grouped into small helper functions so the notebook stays readable.

`load_hybrid_case` prepares the experiment, forcing, observations, and hybrid helpers.
`run_model_param` executes the forward model for one site and one parameter vector.
`analyze_hybrid_case` trains the hybrid model, predicts site parameters, and runs the default and optimized simulations for one selected site.

In [ ]:
function load_hybrid_case(path_experiment_json)
    replace_info = Dict(
        "forcing.subset.site" => selected_site_indices,
        "optimization.optimization_cost_threaded" => false,
        "optimization.optimization_parameter_scaling" => nothing,
        "hybrid.ml_training.fold_path" => nothing,
        "experiment.model_output.path" => path_output,
    )

    info = getExperimentInfo(path_experiment_json; replace_info=deepcopy(replace_info))
    forcing = getForcing(info)
    observations = getObservation(info, forcing.helpers)
    sites_forcing = forcing.data[1].site
    hybrid_helpers = prepHybrid(forcing, observations, info, info.hybrid.ml_training.method)
    return (; info, forcing, observations, sites_forcing, hybrid_helpers, replace_info)
end

function run_model_param(info, forcing, observations, site_index, loc_params)
    run_helpers = prepTEM(info.models.forward, forcing, observations, info)

    selected_models = info.models.forward
    parameter_scaling_type = info.optimization.run_options.parameter_scaling
    tbl_params = info.optimization.parameter_table
    param_to_index = getParameterIndices(selected_models, tbl_params)
    models = updateModels(loc_params, param_to_index, parameter_scaling_type, selected_models)

    loc_forcing = run_helpers.space_forcing[site_index]
    loc_spinup_forcing = run_helpers.space_spinup_forcing[site_index]
    loc_forcing_t = run_helpers.loc_forcing_t
    loc_output = getCacheFromOutput(run_helpers.space_output[site_index], info.hybrid.ml_gradient.method)
    gradient_lib = info.hybrid.ml_gradient.method
    loc_output_from_cache = getOutputFromCache(loc_output, loc_params, gradient_lib)
    land_init = deepcopy(run_helpers.loc_land)
    tem_info = run_helpers.tem_info
    loc_obs = run_helpers.space_observation[site_index]
    loc_cost_option = prepCostOptions(loc_obs, info.optimization.cost_options)
    constraint_method = info.optimization.run_options.multi_constraint_method

    coreTEM!(
        models,
        loc_forcing,
        loc_spinup_forcing,
        loc_forcing_t,
        loc_output_from_cache,
        land_init,
        tem_info,
    )

    forward_output = (; Pair.(getUniqueVarNames(info.output.variables), loc_output_from_cache)...)
    loss_vector = SindbadTutorials.metricVector(loc_output_from_cache, loc_obs, loc_cost_option)
    t_loss = combineMetric(loss_vector, constraint_method)
    return forward_output, loss_vector, t_loss
end

function analyze_hybrid_case(case_name, path_experiment_json; site_index=1)
    case = load_hybrid_case(path_experiment_json)

    trainML(case.hybrid_helpers, case.info.hybrid.ml_training.method)

    ml_model = case.hybrid_helpers.ml_model
    xfeatures = case.hybrid_helpers.features.data
    loss_functions = case.hybrid_helpers.loss_functions
    loss_component_functions = case.hybrid_helpers.loss_component_functions

    params_sites = ml_model(xfeatures)
    @info "params_sites: [$(minimum(params_sites)), $(maximum(params_sites))]"

    scaled_params_sites = getParamsAct(params_sites, case.info.optimization.parameter_table)
    @info "scaled_params_sites: [$(minimum(scaled_params_sites)), $(maximum(scaled_params_sites))]"

    site_name = case.sites_forcing[site_index]
    loc_params = scaled_params_sites(site=site_name).data.data

    loss_f_site = loss_functions(site=site_name)
    loss_vector_f_site = loss_component_functions(site=site_name)
    @time loss_f_site(loc_params)
    loss_vector_site = loss_vector_f_site(loc_params)

    output_default_site, loss_default, total_loss_default = run_model_param(
        case.info,
        case.forcing,
        case.observations,
        site_index,
        case.info.optimization.parameter_table.default,
    )

    output_optimized_site, loss_optimized, total_loss_optimized = run_model_param(
        case.info,
        case.forcing,
        case.observations,
        site_index,
        loc_params,
    )

    return (; case_name, case..., site_index, site_name, ml_model, xfeatures, loss_functions,
        loss_component_functions, params_sites, scaled_params_sites, loc_params, loss_vector_site,
        output_default_site, output_optimized_site, loss_default, loss_optimized,
        total_loss_default, total_loss_optimized)
end

## 4. Train and analyze the WROASTED hybrid setup

The first case uses the `WROASTED_HB` setup. This is the more detailed coupled carbon-water model, so it is the more expensive run but also the one with richer process structure.

The hybrid model is trained first. Then the notebook extracts the learned site-level parameter vector for one selected site and compares the default and hybrid-informed forward simulations.

In [ ]:
path_experiment_json = joinpath(wroasted_setup_dir, "experiment_hybrid.json")
site_index = 67
wroasted_result = analyze_hybrid_case("WROASTED_HB", path_experiment_json; site_index=site_index)
wroasted_result.site_name

## 5. Train and analyze the LUE hybrid setup

The second case uses the LUE-based hybrid setup. It is simpler and faster, which makes it useful for sensitivity tests, quick prototyping, or checking whether a simpler process representation is enough for a given diagnostic target.

In [ ]:
path_experiment_json = joinpath(lue_setup_dir, "experiment_hybrid.json")
lue_result = analyze_hybrid_case("LUE", path_experiment_json; site_index=site_index)
lue_result.site_name

## 6. Posterior diagnostics for a selected site

After the hybrid model predicts site-specific parameters, we can compare forward simulations with:

- the default parameter set, and
- the hybrid-informed parameter set.

The plotting code below follows the original script closely. For each constrained variable, it overlays observations, default simulations, and optimized simulations, and reports the corresponding metric values in the legend.

In [ ]:
function plot_site_diagnostics(result)
    def_dat = result.output_default_site
    opt_dat = result.output_optimized_site
    site_index = result.site_index
    info = result.case.info
    observations = result.case.observations
    site_name = result.site_name

    loc_observation = [Array(o[:, site_index]) for o in observations.data]
    costOpt = prepCostOptions(loc_observation, info.optimization.cost_options)
    default(titlefont=(20, "times"), legendfontsize=18, tickfont=(15, :blue))

    foreach(costOpt) do var_row
        v = var_row.variable
        println("plot obs::", v)
        v = (var_row.mod_field, var_row.mod_subfield)
        vinfo = getVariableInfo(v, info.experiment.basics.temporal_resolution)
        v = vinfo["standard_name"]
        lossMetric = var_row.cost_metric
        loss_name = nameof(typeof(lossMetric))
        if loss_name in (:NNSEInv, :NSEInv)
            lossMetric = NSE()
        end

        (obs_var, obs_σ, def_var) = getData(def_dat, loc_observation, var_row)
        (_, _, opt_var) = getData(opt_dat, loc_observation, var_row)
        obs_var_TMP = obs_var[:, 1, 1, 1]
        non_nan_index = findall(x -> !isnan(x), obs_var_TMP)
        if length(non_nan_index) < 2
            tspan = 1:length(obs_var_TMP)
        else
            tspan = first(non_nan_index):last(non_nan_index)
        end

        obs_σ = obs_σ[tspan]
        obs_var = obs_var[tspan, 1, 1, 1]
        def_var = def_var[tspan, 1, 1, 1]
        opt_var = opt_var[tspan, 1, 1, 1]

        xdata = [info.helpers.dates.range[tspan]...]
        obs_var_n, obs_σ_n, def_var_n = getDataWithoutNaN(obs_var, obs_σ, def_var)
        obs_var_n, obs_σ_n, opt_var_n = getDataWithoutNaN(obs_var, obs_σ, opt_var)
        metr_def = metric(obs_var_n, obs_σ_n, def_var_n, lossMetric)
        metr_opt = metric(obs_var_n, obs_σ_n, opt_var_n, lossMetric)

        p = plot(xdata, obs_var; label="obs", seriestype=:scatter, mc=:black, ms=4, lw=0, ma=0.65, left_margin=1Plots.cm)
        plot!(p, xdata, def_var, color=:steelblue2, lw=1.5, ls=:dash, left_margin=1Plots.cm,
            legend=:outerbottom, legendcolumns=3,
            label="def ($(round(metr_def, digits=2)))", size=(2000, 1000),
            title="$(vinfo["long_name"]) ($(vinfo["units"])) -> $(nameof(typeof(lossMetric)))")
        plot!(p, xdata, opt_var; color=:seagreen3, label="opt ($(round(metr_opt, digits=2)))", lw=1.5, ls=:dash)
        display(p)
        savefig(joinpath(info.output.dirs.figure, "$(result.case_name)_$(site_name)_$(v).png"))
    end
end

In [ ]:
plot_site_diagnostics(wroasted_result)

You can repeat the same diagnostic plots for the LUE case if needed:

In [ ]:
# plot_site_diagnostics(lue_result)

## 7. Histogram of scalar losses across sites

The final part of the original script loops over all sites and compares the total loss of the default versus optimized parameter sets. This is a useful compact summary because it shows whether hybrid-informed parameters improve performance systematically across the training domain, rather than only at one example site.

In [ ]:
function compute_site_loss_histogram(result)
    info = result.case.info
    forcing = result.case.forcing
    observations = result.case.observations
    sites_forcing = result.case.sites_forcing
    scaled_params_sites = result.scaled_params_sites

    n_sites = length(sites_forcing)
    t_loss_def = Vector{Float32}(undef, n_sites)
    t_loss_opt = Vector{Float32}(undef, n_sites)

    for (i, site_index) in enumerate(1:n_sites)
        @info site_index
        @info sites_forcing[site_index]
        @info "running using default parameters..."
        _, _, tl_def = run_model_param(info, forcing, observations, site_index, info.optimization.parameter_table.default)

        @info "running using optimized parameters..."
        _, _, tl_opt = run_model_param(info, forcing, observations, site_index, scaled_params_sites(site=sites_forcing[site_index]).data.data)

        t_loss_def[i] = tl_def
        t_loss_opt[i] = tl_opt
    end

    p = histogram(
        t_loss_def,
        bins=50,
        alpha=0.5,
        label="default",
        xlabel="Loss (1-NSE)",
        ylabel="Count",
        title="Model Loss: default vs optimized",
    )
    histogram!(p, t_loss_opt, bins=50, alpha=0.5, label="optimized")
    savefig(joinpath(info.output.dirs.figure, "loss_histogram_scalar.png"))
    return (; t_loss_def, t_loss_opt, p)
end

In [ ]:
# This can take a long time on the full site set.
# wroasted_hist = compute_site_loss_histogram(wroasted_result)
# display(wroasted_hist.p)

## 8. Interpretation

This hybrid inversion exercise illustrates the central idea of the Sindbad hybrid framework: instead of treating parameters as fixed values or optimizing them independently at each site, we learn a mapping from site-level covariates to plausible parameter sets.

Thinking:
- Which static covariates are most important for predicting site-level parameters? Can we interpret the learned mapping to gain insights into how different environmental conditions influence parameter values?
- How does the performance of the hybrid model compare to the default model across different sites and variables? Are there specific conditions or variables where the hybrid model shows more improvement?
- What are the limitations of this approach? For example, does the hybrid model overfit to the training sites, or does it generalize well to new sites? How sensitive are the results to the choice of covariates or the machine learning model architecture?
- ...